# mf6Voronoi Tutorial

This notebook demonstrates how to use the `mf6Voronoi` package to generate a Voronoi grid for MODFLOW 6 DISV.

In [ ]:
import os
import geopandas as gpd
from shapely.geometry import Polygon, Point, LineString
from mf6Voronoi.voronoi import VoronoiGrid
import numpy as np
import rasterio
from rasterio.transform import from_origin

## 1. Create Synthetic Data

In [ ]:
os.makedirs('data', exist_ok=True)

# Boundary
poly = Polygon([(0, 0), (1000, 0), (1000, 1000), (0, 1000), (0, 0)])
gpd.GeoDataFrame({'geometry': [poly]}, crs="EPSG:32618").to_file('data/boundary.shp')

# Refinement Area
poly_ref = Polygon([(100, 100), (300, 100), (300, 300), (100, 300), (100, 100)])
gpd.GeoDataFrame({'geometry': [poly_ref]}, crs="EPSG:32618").to_file('data/refinement.shp')

# Refinement Line
line = LineString([(500, 0), (500, 1000)])
gpd.GeoDataFrame({'geometry': [line]}, crs="EPSG:32618").to_file('data/river.shp')

# Wells
p1 = Point(800, 800)
gpd.GeoDataFrame({'geometry': [p1]}, crs="EPSG:32618").to_file('data/wells.shp')

# DEM
transform = from_origin(0, 1000, 10, 10)
data = np.zeros((100, 100), dtype=np.float32)
for r in range(100):
    data[r, :] = r
with rasterio.open('data/dem.tif', 'w', driver='GTiff', height=100, width=100, count=1, dtype=data.dtype, crs="EPSG:32618", transform=transform) as dst:
    dst.write(data, 1)

## 2. Configure Voronoi Grid

In [ ]:
vor = VoronoiGrid('data/boundary.shp', min_cell_size=10, max_cell_size=100)

# Add Refinements
vor.add_refinement_area('data/refinement.shp', cell_size=20)
vor.add_refinement_line('data/river.shp', cell_size=20)

# Add Fixed Points (List)
vor.add_fixed_points([(800, 800)])

# Add DEM
vor.add_dem('data/dem.tif', min_slope=0, max_slope=10, min_cell_size=10, max_cell_size=100)

## 3. Build Mesh

In [ ]:
vor.build()

## 4. Visualize

In [ ]:
vor.plot(interactive=True)

## 5. Export

In [ ]:
vor.export_mesh('data/mesh.shp')

disv_props = vor.get_disv_properties()
print("NCPL:", disv_props['ncpl'])